# 自定義層

深度學習成功背後的一個因素是神經網路的靈活性：
我們可以創造性地組合不同的層，從而設計出適用於各種任務的架構。
例如，研究人員發明了專門用於處理圖像、文本、序列數據和執行動態規劃的層。
有時我們會遇到或要自己發明一個現在在深度學習框架中還不存在的層。
在這些情況下，必須構建自定義層。本節將展示如何構建自定義層。

## 不帶參數的層

首先，我們(**構造一個沒有任何參數的自定義層**)。
回憶一下在 :numref:`sec_model_construction`對塊的介紹，
這應該看起來很眼熟。
下面的`CenteredLayer`類要從其輸入中減去均值。
要構建它，我們只需繼承基礎層類並實現前向傳播功能。


In [1]:
import torch
import torch.nn.functional as F
from torch import nn


class CenteredLayer(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, X):
        return X - X.mean()

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


讓我們向該層提供一些數據，驗證它是否能按預期工作。


In [2]:
layer = CenteredLayer()
layer(torch.FloatTensor([1, 2, 3, 4, 5]))

tensor([-2., -1.,  0.,  1.,  2.])

現在，我們可以[**將層作為組件合併到更複雜的模型中**]。


In [3]:
net = nn.Sequential(nn.Linear(8, 128), CenteredLayer())

作為額外的健全性檢查，我們可以在向該網路發送隨機數據後，檢查均值是否為0。
由於我們處理的是浮點數，因為存儲精度的原因，我們仍然可能會看到一個非常小的非零數。


In [4]:
Y = net(torch.rand(4, 8))
Y.mean()

tensor(-3.7253e-09, grad_fn=<MeanBackward0>)

## [**帶參數的層**]

以上我們知道了如何定義簡單的層，下面我們繼續定義具有參數的層，
這些參數可以通過訓練進行調整。
我們可以使用內置函數來創建參數，這些函數提供一些基本的管理功能。
比如管理訪問、初始化、共享、保存和加載模型參數。
這樣做的好處之一是：我們不需要為每個自定義層編寫自定義的序列化程序。

現在，讓我們實現自定義版本的全連接層。
回想一下，該層需要兩個參數，一個用於表示權重，另一個用於表示偏置項。
在此實現中，我們使用修正線性單元作為激活函數。
該層需要輸入參數：`in_units`和`units`，分別表示輸入數和輸出數。


In [5]:
class MyLinear(nn.Module):
    def __init__(self, in_units, units):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(in_units, units))
        self.bias = nn.Parameter(torch.randn(units,))
    def forward(self, X):
        linear = torch.matmul(X, self.weight.data) + self.bias.data
        return F.relu(linear)

接下來，我們實例化`MyLinear`類並訪問其模型參數。


In [6]:
linear = MyLinear(5, 3)
linear.weight

Parameter containing:
tensor([[ 0.2143,  0.6700,  0.8841],
        [-0.6669,  0.7886, -0.0873],
        [ 1.4350, -0.1005,  0.4258],
        [ 0.3860,  0.5293, -0.2505],
        [-0.4857,  1.6320,  0.4033]], requires_grad=True)

我們可以[**使用自定義層直接執行前向傳播計算**]。


In [7]:
linear(torch.rand(2, 5))

tensor([[2.4820, 1.6643, 1.2461],
        [1.2644, 2.3667, 0.9320]])

我們還可以(**使用自定義層構建模型**)，就像使用內置的全連接層一樣使用自定義層。


In [8]:
net = nn.Sequential(MyLinear(64, 8), MyLinear(8, 1))
net(torch.rand(2, 64))

tensor([[0.5715],
        [3.6737]])

## 小結

* 我們可以通過基本層類設計自定義層。這允許我們定義靈活的層，其行為與深度學習框架中的任何現有層不同。
* 在自定義層定義完成後，我們就可以在任意環境和網路架構中調用該自定義層。
* 層可以有局部參數，這些參數可以通過內置函數創建。

## 練習

1. 設計一個接受輸入並計算張量降維的層，它返回$y_k = \sum_{i, j} W_{ijk} x_i x_j$。
1. 設計一個返回輸入數據的傅立葉係數前半部分的層。


[Discussions](https://discuss.d2l.ai/t/1835)


練習一：

1. 設計一個接受輸入並計算張量降維的層，它返回$y_k = \sum_{i, j} W_{ijk} x_i x_j$。

我的回答：





以下是實現一個計算張量降維的自定義層：

```python
import torch
import torch.nn as nn

class TensorDimReduction(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        # 創建三維權重張量 W[i,j,k]
        self.weight = nn.Parameter(
            torch.randn(in_features, in_features, out_features) / (in_features ** 0.5)
        )
    
    def forward(self, x):
        # x shape: (batch_size, in_features)
        # 計算 y_k = sum_{i,j} W_{ijk} x_i x_j
        batch_size = x.shape[0]
        
        # 擴展維度以進行批量計算
        # (batch_size, in_features, 1) * (batch_size, 1, in_features)
        # -> (batch_size, in_features, in_features)
        xx = torch.bmm(x.unsqueeze(2), x.unsqueeze(1))
        
        # (batch_size, in_features, in_features) * (in_features, in_features, out_features)
        # -> (batch_size, out_features)
        out = torch.sum(xx.unsqueeze(3) * self.weight, dim=(1,2))
        
        return out

# 測試代碼
if __name__ == "__main__":
    # 創建層
    layer = TensorDimReduction(in_features=3, out_features=2)
    
    # 測試數據
    x = torch.randn(4, 3)  # 批量大小為4，輸入特徵為3
    
    # 前向傳播
    y = layer(x)
    print(f"輸入形狀: {x.shape}")
    print(f"輸出形狀: {y.shape}")
    print(f"輸出值:\n{y}")
```


這個實現：

1. 核心功能：
- 創建三維權重張量 W[i,j,k]
- 計算 y_k = sum_{i,j} W_{ijk} x_i x_j
- 支持批量處理

2. 技術細節：
- 使用 `torch.bmm` 進行批量矩陣乘法
- 使用 `unsqueeze` 調整維度
- 使用 `sum` 進行降維運算

3. 初始化策略：
- 使用縮放的隨機初始化
- 防止梯度消失/爆炸

4. 優化考慮：
- 使用批量操作提高效率
- 避免使用循環

使用示例：
```python
# 在網路中使用
net = nn.Sequential(
    TensorDimReduction(10, 5),
    nn.ReLU(),
    nn.Linear(5, 1)
)

# 訓練
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(net.parameters())

# 前向傳播
x = torch.randn(32, 10)
y = net(x)
```

這個層可以用於：
- 特徵交互建模
- 高階特徵提取
- 維度降低


練習二：

2. 設計一個返回輸入數據的傅立葉係數前半部分的層。

我的回答：





以下是實現一個返回傅立葉係數前半部分的自定義層：

````python
import torch
import torch.nn as nn

class HalfFFTLayer(nn.Module):
    def __init__(self):
        super().__init__()
        
    def forward(self, x):
        # 計算FFT
        fft_result = torch.fft.fft(x, dim=-1)
        # 只保留前半部分（包含直流分量和正頻率部分）
        half_len = (fft_result.shape[-1] + 1) // 2
        return fft_result[..., :half_len]

# 測試代碼
if __name__ == "__main__":
    # 創建層
    layer = HalfFFTLayer()
    
    # 生成測試數據
    x = torch.randn(4, 8)  # 4個樣本，每個8個點
    
    # 前向傳播
    y = layer(x)
    
    print(f"輸入形狀: {x.shape}")
    print(f"輸出形狀: {y.shape}")
    print(f"輸出值:\n{y}")
    
    # 驗證結果
    print("\n驗證第一個樣本的FFT:")
    full_fft = torch.fft.fft(x[0])
    print(f"完整FFT: {full_fft}")
    print(f"我們的層輸出: {y[0]}")
````


特點說明：

1. 功能：
- 使用 torch.fft.fft 計算傅立葉變換
- 只保留前半部分係數（包含重要頻率信息）
- 支持批量處理

2. 應用場景：
- 頻域特徵提取
- 信號處理
- 數據壓縮

3. 注意事項：
- 輸出是複數值
- 保留了所有重要頻率信息
- 可以用於降維

進階版本（添加更多功能）：
````python
class AdvancedHalfFFTLayer(nn.Module):
    def __init__(self, normalize=True, return_magnitude=True):
        super().__init__()
        self.normalize = normalize
        self.return_magnitude = return_magnitude
        
    def forward(self, x):
        # 計算FFT
        fft_result = torch.fft.fft(x, dim=-1)
        
        # 只保留前半部分
        half_len = (fft_result.shape[-1] + 1) // 2
        half_fft = fft_result[..., :half_len]
        
        if self.return_magnitude:
            # 計算幅度譜
            half_fft = torch.abs(half_fft)
            
        if self.normalize:
            # 歸一化
            half_fft = half_fft / torch.sqrt(torch.sum(half_fft**2, dim=-1, keepdim=True))
            
        return half_fft

# 使用示例
layer = AdvancedHalfFFTLayer(normalize=True, return_magnitude=True)
x = torch.randn(4, 8)
y = layer(x)
print(f"輸入形狀: {x.shape}")
print(f"輸出形狀: {y.shape}")
print(f"輸出值:\n{y}")
````


這個進階版本添加了：
1. 幅度譜計算選項
2. 歸一化選項
3. 更靈活的參數控制

在實際應用中，可以根據需要：
- 調整歸一化方式
- 添加相位信息
- 增加頻率選擇功能
- 添加窗函數處理
